# Lab | Web Scraping

Welcome to the "Books to Scrape" Web Scraping Adventure Lab!

**Objective**

In this lab, we will embark on a mission to unearth valuable insights from the data available on Books to Scrape, an online platform showcasing a wide variety of books. As data analyst, you have been tasked with scraping a specific subset of book data from Books to Scrape to assist publishing companies in understanding the landscape of highly-rated books across different genres. Your insights will help shape future book marketing strategies and publishing decisions.

**Background**

In a world where data has become the new currency, businesses are leveraging big data to make informed decisions that drive success and profitability. The publishing industry, much like others, utilizes data analytics to understand market trends, reader preferences, and the performance of books based on factors such as genre, author, and ratings. Books to Scrape serves as a rich source of such data, offering detailed information about a diverse range of books, making it an ideal platform for extracting insights to aid in informed decision-making within the literary world.

**Task**

Your task is to create a Python script using BeautifulSoup and pandas to scrape Books to Scrape book data, focusing on book ratings and genres. The script should be able to filter books with ratings above a certain threshold and in specific genres. Additionally, the script should structure the scraped data in a tabular format using pandas for further analysis.

**Expected Outcome**

A function named `scrape_books` that takes two parameters: `min_rating` and `max_price`. The function should scrape book data from the "Books to Scrape" website and return a `pandas` DataFrame with the following columns:

**Expected Outcome**

- A function named `scrape_books` that takes two parameters: `min_rating` and `max_price`.
- The function should return a DataFrame with the following columns:
  - **UPC**: The Universal Product Code (UPC) of the book.
  - **Title**: The title of the book.
  - **Price (£)**: The price of the book in pounds.
  - **Rating**: The rating of the book (1-5 stars).
  - **Genre**: The genre of the book.
  - **Availability**: Whether the book is in stock or not.
  - **Description**: A brief description or product description of the book (if available).
  
You will execute this script to scrape data for books with a minimum rating of `4.0 and above` and a maximum price of `£20`. 

Remember to experiment with different ratings and prices to ensure your code is versatile and can handle various searches effectively!

**Resources**

- [Beautiful Soup Documentation](https://www.crummy.com/software/BeautifulSoup/bs4/doc/)
- [Pandas Documentation](https://pandas.pydata.org/pandas-docs/stable/index.html)
- [Books to Scrape](https://books.toscrape.com/)


**Hint**

Your first mission is to familiarize yourself with the **Books to Scrape** website. Navigate to [Books to Scrape](http://books.toscrape.com/) and explore the available books to understand their layout and structure. 

Next, think about how you can set parameters for your data extraction:

- **Minimum Rating**: Focus on books with a rating of 4.0 and above.
- **Maximum Price**: Filter for books priced up to £20.

After reviewing the site, you can construct a plan for scraping relevant data. Pay attention to the details displayed for each book, including the title, price, rating, and availability. This will help you identify the correct HTML elements to target with your scraping script.

Make sure to build your scraping URL and logic based on the patterns you observe in the HTML structure of the book listings!


---

**Best of luck! Immerse yourself in the world of books, and may the data be with you!**

**Important Note**:

In the fast-changing online world, websites often update and change their structures. When you try this lab, the **Books to Scrape** website might differ from what you expect.

If you encounter issues due to these changes, like new rules or obstacles preventing data extraction, don’t worry! Get creative.

You can choose another website that interests you and is suitable for scraping data. Options like Wikipedia, The New York Times, or even library databases are great alternatives. The main goal remains the same: extract useful data and enhance your web scraping skills while exploring a source of information you enjoy. This is your opportunity to practice and adapt to different web environments!

In [12]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin


def scrape_books(min_rating, max_price):

    base_url = "https://books.toscrape.com/"
    current_url = base_url

    books_data = []

    rating_map = {
        "One": 1,
        "Two": 2,
        "Three": 3,
        "Four": 4,
        "Five": 5
    }

    session = requests.Session()

    while current_url:

        print(f"Scraping: {current_url}")

        response = session.get(
            current_url,
            timeout=10
        )

        response.raise_for_status()

        soup = BeautifulSoup(
            response.text,
            "html.parser"
        )

        books = soup.find_all(
            "article",
            class_="product_pod"
        )

        for book in books:

            # -------------------------
            # Title
            # -------------------------
            title = book.h3.a["title"]

            # -------------------------
            # Price
            # -------------------------
            price_text = book.find(
                "p",
                class_="price_color"
            ).text.strip()

            price_text = (
                price_text
                .replace("Â", "")
                .replace("£", "")
            )

            price = float(price_text)

            # -------------------------
            # Rating
            # -------------------------
            rating_class = book.find(
                "p",
                class_="star-rating"
            )["class"]

            rating = rating_map[rating_class[1]]

            # -------------------------
            # FILTER FIRST
            # -------------------------
            if rating < min_rating or price > max_price:
                continue

            # -------------------------
            # Availability
            # -------------------------
            availability = book.find(
                "p",
                class_="instock availability"
            ).text.strip()

            # -------------------------
            # Book URL
            # -------------------------
            book_link = book.h3.a["href"]

            book_link = book_link.replace(
                "../../../",
                ""
            )

            if not book_link.startswith("catalogue/"):
                book_link = "catalogue/" + book_link

            book_url = urljoin(
                base_url,
                book_link
            )

            # -------------------------
            # Individual book page
            # -------------------------
            try:

                book_response = session.get(
                    book_url,
                    timeout=10
                )

                book_response.raise_for_status()

            except requests.RequestException:

                print(f"Could not access: {book_url}")
                continue

            book_soup = BeautifulSoup(
                book_response.text,
                "html.parser"
            )

            # -------------------------
            # UPC
            # -------------------------
            upc = None

            product_info = book_soup.find(
                "table",
                class_="table table-striped"
            )

            if product_info:

                for row in product_info.find_all("tr"):

                    cells = row.find_all("td")

                    if (
                        len(cells) == 2
                        and cells[0].text.strip() == "UPC"
                    ):

                        upc = cells[1].text.strip()
                        break

            # -------------------------
            # Genre
            # -------------------------
            genre = None

            breadcrumb = book_soup.find(
                "ul",
                class_="breadcrumb"
            )

            if breadcrumb:

                links = breadcrumb.find_all("a")

                if len(links) >= 3:
                    genre = links[-1].text.strip()

            # -------------------------
            # Description
            # -------------------------
            description = None

            description_header = book_soup.find(
                "div",
                id="product_description"
            )

            if description_header:

                description_paragraph = (
                    description_header.find_next("p")
                )

                if description_paragraph:
                    description = (
                        description_paragraph.text.strip()
                    )

            # -------------------------
            # Save book
            # -------------------------
            books_data.append({
                "UPC": upc,
                "Title": title,
                "Price (£)": price,
                "Rating": rating,
                "Genre": genre,
                "Availability": availability,
                "Description": description
            })

        # -------------------------
        # Next page
        # -------------------------
        next_page = soup.find(
            "li",
            class_="next"
        )

        if next_page:

            next_link = next_page.find("a")["href"]

            current_url = urljoin(
                current_url,
                next_link

            )

        else:

            current_url = None

    return pd.DataFrame(books_data)

In [13]:
df = scrape_books(4, 20)

print(df)

Scraping: https://books.toscrape.com/
Scraping: https://books.toscrape.com/catalogue/page-2.html
Scraping: https://books.toscrape.com/catalogue/page-3.html
Scraping: https://books.toscrape.com/catalogue/page-4.html
Scraping: https://books.toscrape.com/catalogue/page-5.html
Scraping: https://books.toscrape.com/catalogue/page-6.html
Scraping: https://books.toscrape.com/catalogue/page-7.html
Scraping: https://books.toscrape.com/catalogue/page-8.html
Scraping: https://books.toscrape.com/catalogue/page-9.html
Scraping: https://books.toscrape.com/catalogue/page-10.html
Scraping: https://books.toscrape.com/catalogue/page-11.html
Scraping: https://books.toscrape.com/catalogue/page-12.html
Scraping: https://books.toscrape.com/catalogue/page-13.html
Scraping: https://books.toscrape.com/catalogue/page-14.html
Scraping: https://books.toscrape.com/catalogue/page-15.html
Scraping: https://books.toscrape.com/catalogue/page-16.html
Scraping: https://books.toscrape.com/catalogue/page-17.html
Scraping: 